# Power Gym Membership — Data Understanding & EDA

Dataset ini sintetis untuk demonstrasi capstone, bukan data aktual UMKM.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("../datasets/sample_membership_data.csv")
df = pd.read_csv(DATA)
df.head()

## Business understanding
Project Plan menyebut kendala pencarian data anggota dan verifikasi masa berlaku dari pencatatan manual. Analisis ini memperagakan struktur data dan ringkasan deskriptif.

In [ ]:
print("Shape:", df.shape)
print("
Dtypes:
", df.dtypes)
print("
Missing values:
", df.isna().sum())
print("
Duplicate member IDs:", df.member_id.duplicated().sum())

## Cleaning and feature preparation
Parse date columns, trim strings, check duplicate IDs and invalid date ranges, then derive effective status.

In [ ]:
for col in ["start_date", "end_date", "created_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
for col in ["member_id", "name", "package", "administrative_status"]:
    df[col] = df[col].astype("string").str.strip()
df = df.drop_duplicates(subset="member_id").copy()
df["valid_date_range"] = df["end_date"] >= df["start_date"]
today = pd.Timestamp.today().normalize()
df["effective_status"] = np.where(df.end_date < today, "expired", df.administrative_status)
df[["member_id", "package", "start_date", "end_date", "effective_status", "valid_date_range"]]

## Exploratory data analysis
The following charts summarize this small synthetic sample only.

In [ ]:
print(df.describe(include="all"))
print("
Status counts:
", df.effective_status.value_counts())
print("
Package counts:
", df.package.value_counts())
df.effective_status.value_counts().plot(kind="bar", title="Synthetic membership status")
plt.xlabel("Effective status"); plt.ylabel("Number of records"); plt.tight_layout(); plt.show()

In [ ]:
monthly = df.groupby(df.created_at.dt.to_period("M")).size()
monthly.plot(kind="bar", title="Synthetic registrations by month")
plt.xlabel("Registration month"); plt.ylabel("Records"); plt.tight_layout(); plt.show()

## Findings and limitations
- Dataset has only five synthetic records and cannot represent actual gym operations.
- Status and package summaries demonstrate how membership data can be inspected.
- No retention/churn or predictive modeling is claimed because there is no longitudinal member history or defensible prediction target.
- Replace this dataset with authorized, de-identified UMKM data before drawing business conclusions.